In [ ]:
import re
import pandas as pd
import csv
import os

# pattern: <category="Type">inner text</category>
CAT = re.compile(r'<category="([^"]+)">(.*?)</category>', flags=re.DOTALL)

def extract_entities(text: str):
    # returns list of dicts with entity type and text
    return [{'type': t, 'text': val} for t, val in CAT.findall(text or "")]

def strip_category_tags(text: str):
    # replace the whole tag with just its inner text
    return CAT.sub(r'\2', text or "")

def read_ncbi_corpus(fn):
    records = []
    with open(fn, encoding="utf-8") as fh:
        for line in fh:
            pmid, title, abstract = line.rstrip("\n").split("\t", 2)
            records.append({
                "pmid": pmid,
                "title": title,
                "abstract": abstract,
            })
    
    df = pd.DataFrame(records)

    for col in ("title", "abstract"):
        df[f"{col}_clean"]    = df[col].apply(strip_category_tags)
        df[f"{col}_entities"] = df[col].apply(extract_entities)
    return df

test = read_ncbi_corpus("ncbi/NCBI_corpus_testing.txt")
train = pd.concat([read_ncbi_corpus("ncbi/NCBI_corpus_training.txt"), read_ncbi_corpus("ncbi/NCBI_corpus_development.txt")])

## Few shot search
***

In [ ]:
test_texts = {ent["text"] for ents in test["abstract_entities"] for ent in ents}

train["ent_texts"] = train["abstract_entities"].apply(lambda ents: {ent["text"] for ent in ents})
train["ent_types"] = train["abstract_entities"].apply(lambda ents: {ent["type"] for ent in ents})

mask = train["ent_texts"].apply(lambda s: s.isdisjoint(test_texts))
train_neg = train[mask].copy()
train_neg["num_types"] = train_neg.ent_types.map(len)
train_neg["length"] = train_neg.abstract_clean.map(len)
train_neg = train_neg.sort_values(["num_types","length"], ascending=[False, True])

five_shot = train_neg.head(5)
ten_shot = train_neg.head(10)

# Direct Prompt
***

In [ ]:
direct_prompt = """Annotate the given abstract with the following brackets:
<category="SpecificDisease">[...]</category> A textual string referring to a disease name may refer to a Specific Disease, or a Disease Class. The annotation category Specific Disease is used for those mentions which could be linked to one specific definition that does not include further categorization. 
<category="DiseaseClass">[...]</category> Disease mentions that could be described as a family of many Specific Diseases are annotated with an annotation category called Disease Class. 
<category="CompositeMention">[...]</category> A textual string may refer to two or more separate disease mentions. Such mentions are annotated with the Composite Mention category.
<category="Modifier">[...]</category> A textual string may refer to a disease name, but it may modify a noun phrase, or not be a noun phrase and this is better expressed with the Modifier annotation category. 

Here is the abstract:
<abstract>{abstract_input}</abstract>

Make sure to give your final answer in <abstract></abstract> brackets."""

print(direct_prompt)

# Few-shot prompts

In [ ]:
fewshot_prompt_start = """Annotate the given abstract with the following brackets:
<category="SpecificDisease">[...]</category> A textual string referring to a disease name may refer to a Specific Disease, or a Disease Class. The annotation category Specific Disease is used for those mentions which could be linked to one specific definition that does not include further categorization. 
<category="DiseaseClass">[...]</category> Disease mentions that could be described as a family of many Specific Diseases are annotated with an annotation category called Disease Class. 
<category="CompositeMention">[...]</category> A textual string may refer to two or more separate disease mentions. Such mentions are annotated with the Composite Mention category.
<category="Modifier">[...]</category> A textual string may refer to a disease name, but it may modify a noun phrase, or not be a noun phrase and this is better expressed with the Modifier annotation category. 

Here are examples:
"""

fewshot_prompt_end = """Now, apply the same reasoning to this new abstract (and only this abstract):
<abstract>{abstract_input}</abstract>

Make sure to give your final answer in <abstract></abstract> brackets."""

def generate_fewshot_prompt_template(df):
    # uses the rows in df to create few shot examples
    p = fewshot_prompt_start
    for i, row in df.iterrows():
        p += "**INPUT**: <abstract>" + row.abstract_clean + "</abstract>\n**OUTPUT**: <abstract>" + row.abstract + "</abstract>\n\n"
    p += fewshot_prompt_end
    return p

five_shot_prompt = generate_fewshot_prompt_template(five_shot)
ten_shot_prompt = generate_fewshot_prompt_template(ten_shot)

# Mechanism Prompt
***

In [ ]:
mechanism_prompt = """Please follow the annotation guideline below and apply it on the given abstract.

An Annotation Guideline for Disease Mentions in Abstracts

This document provides instructions for annotating four categories of disease-related mentions in scientific abstracts. The goal is to accurately identify and classify textual strings that refer to specific diseases, classes of diseases, multiple mentions, and disease modifiers.

### General Instructions

Read the abstract carefully and identify all mentions related to diseases or medical conditions. For each mention, select the most appropriate category from the four options below and enclose the textual string in the corresponding tags.

---

### 1. Specific Disease

**Tag:** `<category="SpecificDisease">[...]</category>`

**Definition:** This category is for disease mentions that refer to a single, specific medical condition that has a well-defined meaning and does not represent a broader classification.

**Rules:**

*   **Specificity:** The term should be a concrete diagnosis. For example, "van der Woude syndrome" is a specific condition, whereas "inherited disorder" is not.
*   **Noun Form:** The mention should typically be a noun or noun phrase representing the disease itself.
*   **Acronyms:** When a disease is defined with its acronym (e.g., "Aarskog-Scott Syndrome (AAS)"), both the full name and the acronym should be tagged separately as `SpecificDisease`.

---

### 2. Disease Class

**Tag:** `<category="DiseaseClass">[...]</category>`

**Definition:** This category is used for disease mentions that describe a family, group, or general type of disease, rather than a single specific one.

**Rules:**

*   **Generality:** The term should be a broad classification that could contain multiple specific diseases.
*   **Hierarchy:** Think of a `DiseaseClass` as a parent category. For example, `dystonia` is a `DiseaseClass`, while `oromandibular dystonia` is a `SpecificDisease` that falls under it.
*   **Symptoms vs. Classes:** General symptoms or findings that represent a class of conditions (e.g., "muscular atrophy") should be tagged as `DiseaseClass`.

---

### 3. Composite Mention

**Tag:** `<category="CompositeMention">[...]</category>`

**Definition:** This category is for a single textual string that explicitly lists two or more distinct disease mentions.

**Rules:**

*   **Conjunctions:** Look for coordinating conjunctions like "and," "or," "and/or," or a series of commas that link multiple conditions.
*   **Span:** The annotation must cover the entire phrase, including all individual disease names and the words connecting them.

---

### 4. Modifier

**Tag:** `<category="Modifier">[...]</category>`

**Definition:** This category is for a textual string where a disease name is used adjectivally to modify another noun, rather than referring to the disease itself.

**Rules:**

*   **Adjectival Function:** The disease term is not the subject but describes something else. It often answers the question "what kind?" or "which one?".
*   **Context:** The disease name will typically precede nouns such as "patients," "gene," "symptoms," "phenotype," "group," "families," or "region."

Here is the abstract:
<abstract>{abstract_input}</abstract>"""

# Hybrid
***

In [ ]:
hybrid_prompt = """Here is an example annotation process that I need you to apply on a new abstract (given afterwards):

INPUT: <abstract>Glucose-6-phosphate dehydrogenase ( G6PD ) deficiency in RBCs was found significantly more frequently in 210 male cataractous patients than in 672 control subjects of Sardinian origin . The frequency of the deficiency was increasingly higher in presenile cataracts . In the G6PD-deficient group , the incidence of cortical and total cataracts was also increased . It is suggested that decrease of the G6PD activity in the lens , which accompanies its deficiency in the erythrocyte , might play a role in the cataracto-genesis of these patients . Moreover , G6PD deficiency should be added to other conditions , such as the galactosemic states and riboflavin deficiency , where cataracts represent a sensitive indicator of metabolic abnormalities of the RBC . . </abstract>

**Step 1: Identify All Potential Disease-Related Terms**
Okay, I found the following terms related to disease:
1. "Glucose-6-phosphate dehydrogenase ( G6PD )"
2. "cataractous"
3. "deficiency"
4. "presenile"
5. "cataracts"
6. "G6PD-deficient"
7. "cortical and total cataracts"
8. "G6PD activity"
9. "G6PD deficiency"
10. "galactosemic"
11. "riboflavin deficiency"
12. "metabolic abnormalities"

**Step 2: Classify Each Identified Mention**
Next, I need to classify each identified mention into one of the four categories or discard it if it does not fit.
1.  **"Glucose-6-phosphate dehydrogenase (G6PD)"**: While G6PD is an enzyme, the term is frequently used as a shorthand to refer to the disease caused by its deficiency. In this context, it functions as a specific disease name that can be linked to a single, well-defined condition. Therefore, it is classified as a *SpecificDisease*.

2.  **"cataractous"**: This term is an adjective used to describe a noun (e.g., "cataractous lens"). It modifies another word to indicate the presence of cataracts but does not represent the disease itself in this form. For this reason, it is classified as a *Modifier*.

3.  **"deficiency"**: Standing alone, "deficiency" is a general term that does not name a specific disease. It describes a state of lacking something and requires more context (e.g., "iron deficiency") to become a diagnosable condition. Thus, on its own, it is *discarded*.

4.  **"presenile"**: This is an adjective that describes the onset of a condition as occurring earlier than is typical, but it is not a disease itself. It modifies a condition (e.g., "presenile dementia") but is too general to be a modifier of a specific disease entity in the way other terms are. Therefore, it is *discarded*.

5.  **"cataracts"**: This refers to a specific medical condition involving the clouding of the lens in the eye. Although there are different types, "cataracts" itself is considered a single, specific diagnosis. It can be linked to a clear definition and is not a broad family of diseases. This makes it a *SpecificDisease*.

6.  **"G6PD-deficient"**: This is an adjectival phrase used to describe a person or cell that has a deficiency of the G6PD enzyme. It modifies a noun (e.g., "a G6PD-deficient patient") rather than being the noun phrase for the disease itself. Consequently, it is classified as a *Modifier*.

7.  **"cortical and total cataracts"**: This phrase explicitly names two distinct types of cataracts ("cortical" and "total"). The use of "and" indicates that it is referring to two separate disease mentions joined together. This fits the definition of a *CompositeMention*.

8.  **"G6PD activity"**: This phrase refers to the function or measurement of the G6PD enzyme. It describes a biological process or a laboratory value, not a disease or a health condition. For this reason, it is *discarded*.

9.  **"G6PD deficiency"**: This is the full and specific name of a recognized medical disorder. It refers to a single, definable inherited condition, which makes it a *SpecificDisease*.

10. **"galactosemic"**: This is an adjective derived from the disease "galactosemia." It is used to modify a noun, for instance, a "galactosemic patient." Since it describes a noun in the context of a disease, it is classified as a *Modifier*.

11. **"riboflavin deficiency"**: This term refers to a specific nutritional disorder with a well-defined cause (lack of riboflavin) and a specific set of symptoms. It is considered a single, distinct medical condition. Therefore, it is classified as a *SpecificDisease*.

12. **"metabolic abnormalities"**: This is a broad, high-level term that encompasses a wide range of individual diseases and disorders related to metabolism (e.g., diabetes, phenylketonuria). Because it represents a family or class of many specific diseases, it is classified as a *DiseaseClass*.

The final list of mentions is thus:
1.  **"Glucose-6-phosphate dehydrogenase ( G6PD )"**: SpecificDisease
2.  **"cataractous"**: Modifier
3.  **"deficiency"**: discard
4.  **"presenile"**: discard
5.  **"cataracts"**: SpecificDisease
6.  **"G6PD-deficient"**: Modifier
7.  **"cortical and total cataracts"**: CompositeMention
8.  **"G6PD activity"**: discard
9.  **"G6PD deficiency"**: SpecificDisease
10. **"galactosemic"**: modifier
11. **"riboflavin deficiency"**: SpecificDisease
12. **"metabolic abnormalities"**: DiseaseClass

**Step 3: Apply Annotation Formatting**
Next, I need to apply the formatting with the correct XML-style tags. The format is <category="[CategoryName]">[term or phrase]</category>, where [CategoryName] is one of the four defined categories.
So, we get:
<abstract><category="SpecificDisease">Glucose-6-phosphate dehydrogenase ( G6PD ) deficiency</category> in RBCs was found significantly more frequently in 210 male <category="Modifier">cataractous</category> patients than in 672 control subjects of Sardinian origin . The frequency of the deficiency was increasingly higher in <category="Modifier">presenile</category> <category="SpecificDisease">cataracts</category> . In the <category="Modifier">G6PD-deficient</category> group , the incidence of <category="CompositeMention">cortical and total cataracts</category> was also increased . It is suggested that decrease of the G6PD activity in the lens , which accompanies its deficiency in the erythrocyte , might play a role in the cataracto-genesis of these patients . Moreover , <category="SpecificDisease">G6PD deficiency</category> should be added to other conditions , such as the <category="Modifier">galactosemic</category> states and <category="SpecificDisease">riboflavin deficiency</category> , where <category="SpecificDisease">cataracts</category> represent a sensitive indicator of <category="DiseaseClass">metabolic abnormalities</category> of the RBC . . </abstract>

**Step 4: Conduct a Final Review**
Now, I need to check whether I correctly annotated all mentions.
1.  **"Glucose-6-phosphate dehydrogenase ( G6PD )"**: SpecificDisease. --> Correct.
2.  **"cataractous"**: Modifier. --> Correct.
3.  **"deficiency"**: discard. --> Correct.
4.  **"presenile"**: discard. --> Incorrect. This should have been discarded!
5.  **"cataracts"**: SpecificDisease. --> Correct.
6.  **"G6PD-deficient"**: Modifier. --> Correct.
7.  **"cortical and total cataracts"**: CompositeMention. --> Correct.
8.  **"G6PD activity"**: discard. --> Correct.
9.  **"G6PD deficiency"**: SpecificDisease. --> Correct.
10. **"galactosemic"**: modifier. --> Correct.
11. **"riboflavin deficiency"**: SpecificDisease. --> Correct.
12. **"metabolic abnormalities"**: DiseaseClass. --> Correct.

After careful review, most mentions were annotated correctly, with one exception.
The revised abstract is thus:
<abstract><category="SpecificDisease">Glucose-6-phosphate dehydrogenase ( G6PD ) deficiency</category> in RBCs was found significantly more frequently in 210 male <category="Modifier">cataractous</category> patients than in 672 control subjects of Sardinian origin . The frequency of the deficiency was increasingly higher in presenile <category="SpecificDisease">cataracts</category> . In the <category="Modifier">G6PD-deficient</category> group , the incidence of <category="CompositeMention">cortical and total cataracts</category> was also increased . It is suggested that decrease of the G6PD activity in the lens , which accompanies its deficiency in the erythrocyte , might play a role in the cataracto-genesis of these patients . Moreover , <category="SpecificDisease">G6PD deficiency</category> should be added to other conditions , such as the <category="Modifier">galactosemic</category> states and <category="SpecificDisease">riboflavin deficiency</category> , where <category="SpecificDisease">cataracts</category> represent a sensitive indicator of <category="DiseaseClass">metabolic abnormalities</category> of the RBC . . </abstract>

**Step 5: Format your final answer**
Did I enclose my final answer with <abstract></<abstract>>? Yes.

----- END OF EXAMPLE -----

Now, please apply the same logic and reasoning to this abstract, and this abstract only:
<abstract>{abstract_input}</abstract>"""

# AI generated mechanism
***

In [ ]:
mechanism_ai_prompt = """Please follow the annotation guideline below and apply it on the given abstract.

### **Guideline for Annotating Disease Mentions**

**Objective**: To annotate textual mentions of diseases in medical abstracts using four categories: `SpecificDisease`, `DiseaseClass`, `CompositeMention`, and `Modifier`.

**General Procedure**:
1.  First, scan the abstract to identify all potential phrases related to diseases, disorders, syndromes, symptoms, or pathological conditions.
2.  For each identified phrase, apply the following rules in order to determine the correct category. The rules are hierarchical; the first rule that matches a mention should be used.

---

### **Rule 1: Is the mention a Modifier?**

A disease mention is a `Modifier` if it is used to describe another noun, rather than referring to the disease itself as the primary subject.

*   **Primary Test**: Check if the disease term acts as an adjective. This can be identified in two main ways:
    1.  **Adjectival Form**: The word is an adjective derived from a disease name (e.g., `cataractous` patients, `dystonic` symptoms).
    2.  **Noun Adjunct**: A disease name (or its acronym) is placed directly before another noun to specify its type.
*   **Annotation Rule**: Tag only the disease term(s), not the noun being modified.
    *   If multiple disease names modify a noun (e.g., `X-ALD / AMN Study Group`), tag each one separately as a modifier: .
    *   If a multi-word phrase acts as a single modifier, tag the whole phrase.

**If the mention is a Modifier, annotate it as `<category="Modifier">[...]</category>` and move to the next mention. If not, proceed to Rule 2.**

---

### **Rule 2: Is the mention a Composite Mention?**

A mention is a `CompositeMention` if it explicitly lists two or more distinct diseases, symptoms, or abnormalities.

*   **Look for Lists**: These phrases are typically lists joined by conjunctions like "and", "or", slashes ("/"), or commas.
*   **Annotation Rule**: The tag must encompass the entire list, including the names of the conditions and the connecting words or punctuation that form the list structure.

**If the mention is a Composite Mention, annotate it as `<category="CompositeMention">[...]</category>` and move to the next mention. If not, proceed to Rule 3.**

---

### **Rule 3: Is the mention a Specific Disease or a Disease Class?**

At this point, the mention is a noun phrase referring directly to a condition. The final step is to determine its specificity.

*   **`SpecificDisease`**: Use this for mentions that refer to a single, well-defined pathological condition that does not represent a broader category.
    *   **Clues**:
        *   Named syndromes (e.g., `van der Woude syndrome`, `McLeod syndrome`).
        *   Precise medical conditions (e.g., `hypodontia`, `cleft palate`, `acanthocytosis`, `cataracts`).
        *   Specific subtypes of a broader class (e.g., `oromandibular dystonia` is a specific type of the class `dystonia`).
        *   Acronyms for specific diseases when used as nouns.
    *   **Special Case**: When a disease and its acronym are introduced together, tag both the full name and the acronym separately as `SpecificDisease`.

*   **`DiseaseClass`**: Use this for mentions that describe a general category or family of diseases, rather than one single entity.
    *   **Clues**:
        *   General, descriptive words like `disorder`, `abnormalities`, `condition`, `dystrophy`, `atrophy`, `demyelination`, `clefts`, `error of metabolism`.
        *   The term represents a group that contains multiple, more specific diseases. For example, `dystonia` is a class of movement disorders.
    *   **Annotation Rule**: Include any preceding adjectives that help define the class (e.g., tag the full phrase `inherited disorder` or `metabolic abnormalities`).

**Apply either `<category="SpecificDisease">[...]</category>` or `<category="DiseaseClass">[...]</category>` based on this distinction.**

Here is the abstract:
<abstract>{abstract_input}</abstract>

Make sure to give your final answer in <abstract></abstract> brackets."""

# Constructing the files
***

In [ ]:
prompt_types = dict(P_X=direct_prompt, P_E5=five_shot_prompt, P_E10=ten_shot_prompt, P_H=hybrid_prompt, P_MAI=mechanism_ai_prompt, P_M=mechanism_prompt)

In [ ]:
def create_prompt_dicts(row):
    dicts = []
    for prompt_type, prompt_template in prompt_types.items():
        p = prompt_template.format(abstract_input=row.abstract_clean)
        d = dict(custom_id=f"{row.pmid}_{prompt_type}", method="POST",url="/v1/chat/completions", 
                 body=dict(model="X", messages=[dict(role="user",content=p)]))
        dicts.append(d)
    return dicts

requests = test.apply(create_prompt_dicts , axis=1)

In [ ]:
requests.explode().to_json("data/ncbi.jsonl", orient="records", lines=True, force_ascii=False)